# Task 5 — Analytics Report & Insights Documentation
## Titanic Passenger Outcomes: A Data-Driven Safety and Equity Review

**Author:** Monika  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Dataset:** `train.csv` — Titanic training dataset (891 passenger records)

### Business / decision objective
Use passenger-level data to describe survival outcomes and identify which recorded passenger characteristics were associated with different survival rates. The goal is to demonstrate a transparent analytics workflow and produce evidence-based, appropriately qualified recommendations for passenger-safety planning.

> **Important limitation:** This is a historical, observational dataset. Associations are not proof of causation and the findings should not be generalized to modern transport operations without external validation.


## 1. Load the data and define the questions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("train.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Place train.csv in the same folder as this notebook, then rerun.")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())


In [ ]:
business_questions = [
    "What proportion of passengers survived?",
    "How did observed survival rates differ by sex and passenger class?",
    "How complete are the fields used in the analysis?",
    "Which numeric variables show associations with survival?",
]
for q in business_questions:
    print("•", q)


## 2. Data quality and preparation

In [ ]:
print("Rows and columns:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count").assign(
    missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2)
))
print("Exact duplicate rows:", df.duplicated().sum())


In [ ]:
# Keep the original dataframe unchanged and create an analysis copy.
analysis = df.copy()
analysis["Sex"] = analysis["Sex"].astype("category")
analysis["Pclass"] = analysis["Pclass"].astype("category")
analysis["SurvivalLabel"] = analysis["Survived"].map({0: "Did not survive", 1: "Survived"})
analysis["FamilySize"] = analysis["SibSp"].fillna(0) + analysis["Parch"].fillna(0) + 1

# Report missingness explicitly. For group-rate calculations, Pandas excludes missing outcome values.
print("Missing Age:", analysis["Age"].isna().sum())
print("Missing Embarked:", analysis["Embarked"].isna().sum())
print("Missing Cabin:", analysis["Cabin"].isna().sum())
print("Analysis columns added: SurvivalLabel, FamilySize")


## 3. Executive metrics and descriptive statistics

In [ ]:
n = len(analysis)
survivors = int(analysis["Survived"].sum())
survival_rate = analysis["Survived"].mean() * 100
print(f"Passengers: {n:,}")
print(f"Survivors: {survivors:,}")
print(f"Overall observed survival rate: {survival_rate:.2f}%")
display(analysis[["Age", "Fare", "SibSp", "Parch", "FamilySize"]].describe().T)


## 4. Visualization 1 — Overall outcomes

In [ ]:
outcomes = analysis["SurvivalLabel"].value_counts().reindex(["Did not survive", "Survived"])
ax = outcomes.plot(kind="bar", figsize=(8, 5), rot=0)
ax.set_title("Passenger Outcomes")
ax.set_xlabel("Outcome")
ax.set_ylabel("Passenger count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.show()
display(outcomes.to_frame("passenger_count").assign(percent=lambda x: (x.passenger_count/n*100).round(2)))


**Interpretation:** Compare the survivor and non-survivor counts and their percentages. The commonly reported training-set result is 342 survivors out of 891 passengers (38.38%); 549 passengers did not survive (61.62%). This imbalance means that a model predicting “did not survive” for every passenger would achieve 61.6% accuracy while being useless for identifying survivors. Accuracy alone would therefore be an inadequate evaluation metric for a predictive extension.


## 5. Visualization 2 — Survival rate by sex

In [ ]:
sex_rates = analysis.groupby("Sex", observed=False)["Survived"].agg(["count", "sum", "mean"])
sex_rates["survival_rate_pct"] = sex_rates["mean"] * 100
display(sex_rates[["count", "sum", "survival_rate_pct"]].rename(
    columns={"count":"passengers", "sum":"survivors"}
).round(2))
ax = (sex_rates["survival_rate_pct"]).plot(kind="bar", figsize=(8,5), rot=0)
ax.set_title("Observed Survival Rate by Sex")
ax.set_xlabel("Recorded sex")
ax.set_ylabel("Survival rate (%)")
ax.set_ylim(0,100)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.show()


**Interpretation:** The dataset shows a large observed difference between female and male survival rates: approximately 74.2% for females (233/314) and 18.9% for males (109/577). The gap is descriptive, not causal. It may reflect evacuation practices, social norms, passenger class, age, family structure, and other factors. A stronger follow-up would compare sex within each passenger class and age group rather than relying only on an overall split.


## 6. Visualization 3 — Survival rate by passenger class

In [ ]:
class_rates = analysis.groupby("Pclass", observed=False)["Survived"].agg(["count", "sum", "mean"])
class_rates["survival_rate_pct"] = class_rates["mean"] * 100
display(class_rates[["count", "sum", "survival_rate_pct"]].round(2))
ax = class_rates["survival_rate_pct"].plot(kind="bar", figsize=(8,5), rot=0)
ax.set_title("Observed Survival Rate by Passenger Class")
ax.set_xlabel("Passenger class (1 = first, 3 = third)")
ax.set_ylabel("Survival rate (%)")
ax.set_ylim(0,100)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.show()


**Interpretation:** Observed survival rates are approximately 63.0% in first class (136/216), 47.3% in second class (87/184), and 24.2% in third class (119/491). The pattern indicates an association between passenger class and survival in this sample. Passenger class can be related to cabin location, access routes, fare, and other socioeconomic factors, so it should not be interpreted as a standalone causal explanation.


## 7. Age, fare, and family structure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.histplot(data=analysis, x="Age", hue="SurvivalLabel", bins=25, kde=True,
             stat="count", common_norm=False, ax=axes[0])
axes[0].set_title("Age Distribution by Outcome")
sns.boxplot(data=analysis, x="SurvivalLabel", y="Fare", ax=axes[1])
axes[1].set_title("Fare Distribution by Outcome")
axes[1].set_xlabel("Outcome")
plt.tight_layout()
plt.show()

family_rates = analysis.groupby("FamilySize", observed=True)["Survived"].agg(
    passengers="size", survival_rate="mean"
)
family_rates["survival_rate_pct"] = family_rates["survival_rate"] * 100
display(family_rates[["passengers", "survival_rate_pct"]].round(2))


**Interpretation:** Age and fare distributions should be reviewed for skew, overlapping groups, and outliers. Fare is typically right-skewed, so the mean may be pulled upward by a small number of expensive tickets; median and quartiles are useful complements. Family-size group rates can be unstable for rare group sizes, so always display the passenger count beside a rate and avoid strong conclusions from tiny groups.


## 8. Correlation analysis — numerical variables

In [ ]:
numeric = analysis[["Survived", "Age", "SibSp", "Parch", "Fare", "FamilySize", "Pclass"]].copy()
numeric["Pclass"] = pd.to_numeric(numeric["Pclass"], errors="coerce")
corr = numeric.corr(numeric_only=True)
plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap — Selected Numeric Variables")
plt.tight_layout()
plt.show()
display(corr["Survived"].drop("Survived").sort_values(key=lambda s: s.abs(), ascending=False).to_frame("correlation_with_survival"))


**Heatmap interpretation guide:** Values close to +1 indicate positive linear association; values close to −1 indicate negative linear association; values near 0 indicate weak linear association. The sign of the Pclass correlation requires care because a higher numeric code means a lower passenger class (third class is coded 3). Correlation does not capture every nonlinear relationship and does not establish causation. Sex is categorical and is not represented in this numeric-only heatmap.


## 9. Findings and business implications

1. **Overall survival was low in this sample:** 342 of 891 passengers survived (38.38%). This provides context for the severity of the event but is not a benchmark for modern transport.
2. **Survival differed substantially by recorded sex:** the descriptive rates were about 74.2% for females and 18.9% for males. This warrants stratified analysis and historical context rather than a simplistic causal claim.
3. **Passenger class was associated with survival:** first-class passengers had a higher observed rate than second- and third-class passengers. Class may proxy for several structural factors.
4. **Data completeness varies by field:** Age and especially Cabin contain missing values in the commonly used dataset. Missingness itself can bias analysis if complete cases differ from incomplete cases.
5. **Metrics need context:** passenger counts, denominators, missingness, and uncertainty should accompany percentages. Small groups can yield volatile rates.


## 10. Recommendations

- **For analytics reporting:** show counts and rates together, document denominators, and include missingness summaries.
- **For safety-oriented planning (as a general lesson, not a claim about this historical event):** audit evacuation outcomes by location, mobility needs, age group, and access to exits; verify that procedures are equitable and understandable.
- **For future analysis:** stratify survival by sex and class together; inspect age groups and family size; report confidence intervals for group rates.
- **For predictive work:** use a train/test split, cross-validation, and metrics suited to class imbalance (recall, precision, F1, ROC-AUC); compare against a simple baseline and check subgroup performance.
- **For data governance:** preserve raw data, document transformations, record software versions, and make the notebook reproducible.


## 11. Limitations

- The dataset is historical, small, and not representative of modern passenger populations.
- Several fields contain missing values; missingness may not be random.
- The analysis is observational and cannot prove causes.
- Group comparisons may be confounded by age, class, family composition, and other unmeasured variables.
- Rates for small groups can be unstable; confidence intervals and sensitivity checks should be added for formal decision-making.


## 12. Conclusion

This report demonstrates a reproducible workflow for converting passenger-level data into descriptive metrics, visual evidence, interpretations, and practical next steps. The clearest patterns are the overall survival rate of 38.38%, a substantial sex-based difference in observed rates, and a gradient across passenger classes. These results should be presented as historical associations with explicit limitations, not as causal claims. The notebook computes values from the local `train.csv`, allowing the report to be regenerated and audited.


## Appendix — reproducibility checklist

- Keep `train.csv` in the same directory as this notebook.
- Install dependencies from `requirements.txt`.
- Run all notebook cells from top to bottom.
- Check that displayed counts match the dataset version being analyzed.
- Export charts and record any manual changes to the analysis.
